# Software Discovery

Query the **SKA SRCNet Software Discovery TAP service** using `astroquery.srcnet`.

```
SoftwareDiscovery
  ├─▶ .query_software(status, science_category, …)   # keyword filters
  ├─▶ .get_software(uri)                             # single entry by URI
  ├─▶ .query_by_image(image)                         # search by image name
  ├─▶ .query(adql)                                   # raw ADQL
  ├─▶ .get_columns()                                 # table schema
  ├─▶ .nl_to_adql(text)                              # NL → ADQL translation
  └─▶ .query_natural(text)                           # NL → ADQL + execute
```

All methods return an `astropy.table.Table` — renders as a rich HTML table in Jupyter.  
The TAP table is **`sdm.software`** (normalized multi-table schema).

## 1 · Import

In [ ]:
from astroquery.srcnet import SoftwareDiscovery, SRCNet

## 2 · Inspect the table schema

In [ ]:
SoftwareDiscovery.get_columns()

## 3 · List all registered software

In [ ]:
t = SoftwareDiscovery.query_software(columns="uri, status, description")
t

## 4 · Filtered query

All keyword parameters are optional — combine as many as needed.

In [ ]:
t = SoftwareDiscovery.query_software(
    status           = "STABLE",
    science_category = "Continuum Science",
    # function_category     = "Source Extraction",
    # science_working_group = "Continuum Survey",
    # requires_gpu          = False,
)
t

## 5 · GPU-required software

In [ ]:
t = SoftwareDiscovery.query_software(requires_gpu=True)
t

## 6 · Fetch one entry by URI

In [ ]:
t = SoftwareDiscovery.get_software("ska:sextractor:docker-sextractor@2.25.0")
t

## 7 · Search by Docker image name

In [ ]:
t = SoftwareDiscovery.query_by_image("wsclean")
t[["uri", "location", "cpu_architecture"]]

## 8 · Raw ADQL query

For queries that span multiple tables, use `.query(adql)` directly.  
The schema is normalized — see the docstring for the JOIN patterns.

In [ ]:
t = SoftwareDiscovery.query("""
    SELECT DISTINCT s.uri, s.status, s.description, r.min_memory, r.recommended_memory
    FROM sdm.software AS s
    LEFT JOIN sdm.resource_requirements AS r ON r.software_id = s.id
    WHERE r.requires_gpu = FALSE
      AND r.min_memory <= 4
    ORDER BY s.uri
""")
t

## 9 · Natural language → ADQL

`nl_to_adql()` translates a plain-English question into ADQL using the SRCNet
remote chat service — no local model setup required.

`query_natural()` does the translation **and** executes the query.

In [ ]:
# Translate only — inspect the ADQL before running
adql = SoftwareDiscovery.nl_to_adql("list all stable software that requires a GPU")
print(adql)

In [ ]:
# Translate and execute in one call
adql, t = SoftwareDiscovery.query_natural(
    "show all Docker images for continuum imaging, sorted by URI",
    verbose=True,   # prints the generated ADQL
)
t

In [ ]:
# Aggregate queries
adql, t = SoftwareDiscovery.query_natural(
    "how many entries are there per status category?",
    verbose=True,
)
t

## 10 · Conversational interface

For multi-turn questions, use the chat — it preserves context between calls.

In [ ]:
# Print usage examples
SRCNet.chat()

In [ ]:
t = SRCNet.chat("What software is registered for spectral-line processing?")

In [ ]:
# Follow-up — context is preserved
t = SRCNet.chat("Which of those have a Docker image available?")

In [ ]:
# Start a fresh session
SRCNet._chat.reset()

## 11 · Async / long-running queries (optional)

For very large result sets, submit an async TAP job via pyvo directly.  
The `.tap` property exposes the underlying `pyvo.dal.TAPService`.

In [ ]:
job = SoftwareDiscovery.tap.submit_job(
    "SELECT * FROM sdm.software",
    maxrec=10_000,
)
job.run()
job.wait(phases=["COMPLETED", "ERROR", "ABORTED"])
job.raise_if_error()

t = job.fetch_result().to_table()
print(f"{len(t)} rows retrieved")
t